# Prediction and Forecasting

This notebook demonstrates how to load trained models and generate predictions with confidence intervals.

**Requirements covered:** 6.1-6.6, 9.1-9.5

## Objectives
1. Load trained model from registry
2. Generate single-step predictions
3. Generate multi-step forecasts
4. Calculate confidence intervals
5. Visualize predictions with uncertainty
6. Demonstrate different forecast horizons

In [ ]:
import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from src.data_ingestion import DataIngestionManager
from src.data_preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer
from src.dataset_splitter import DatasetSplitter
from src.prediction_service import PredictionService
from src.model_registry import ModelRegistry
from src.visualization_manager import VisualizationManager
from config import Config

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Load Trained Model

In [ ]:
# Initialize model registry and prediction service
registry = ModelRegistry()
predictor = PredictionService()

# List available models
models = registry.list_models()

print("Available models:")
print("="*80)
for model_info in models:
    print(f"Version: {model_info.version}")
    print(f"  Type: {model_info.model_type}")
    print(f"  Training Date: {model_info.training_date}")
    print(f"  Performance: RMSE={model_info.performance_metrics.get('rmse', 'N/A'):.4f}")
    print("-"*80)

In [ ]:
# Load the latest model
try:
    model, metadata = registry.load_latest_model()
    print(f"Loaded model: {metadata.version}")
    print(f"Model type: {metadata.model_type}")
    print(f"Features: {len(metadata.feature_list)}")
    print(f"Training data: {metadata.training_data_range[0]} to {metadata.training_data_range[1]}")
    print(f"\nPerformance metrics:")
    for metric, value in metadata.performance_metrics.items():
        print(f"  {metric}: {value:.4f}")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please run the 02_model_training.ipynb notebook first to train a model.")

## 2. Prepare Input Data

In [ ]:
# Load and preprocess data
data_manager = DataIngestionManager()
gold_df = data_manager.load_csv('../XAU_1d_data.csv')

print(f"Loaded {len(gold_df)} records")

# Load indicators
start_date = gold_df['Date'].min().strftime('%Y-%m-%d')
end_date = gold_df['Date'].max().strftime('%Y-%m-%d')
tickers = list(Config.INDICATORS.values())

try:
    indicators = data_manager.load_economic_indicators(tickers, start_date, end_date)
except:
    indicators = {}

# Preprocess
preprocessor = DataPreprocessor()
gold_df = preprocessor.handle_missing_values(gold_df)
gold_df = preprocessor.remove_outliers(gold_df)

if indicators:
    combined_df = preprocessor.align_datasets(gold_df, indicators)
else:
    combined_df = gold_df

print(f"Preprocessed: {len(combined_df)} records")

In [ ]:
# Engineer features (same as training)
feature_engineer = FeatureEngineer()

featured_df = feature_engineer.create_lag_features(combined_df, 'Close', Config.LAG_PERIODS)
featured_df = feature_engineer.create_rolling_features(featured_df, 'Close', Config.ROLLING_WINDOWS)
featured_df = feature_engineer.create_technical_indicators(featured_df)
featured_df = feature_engineer.create_temporal_features(featured_df)

if indicators:
    featured_df = feature_engineer.create_interaction_features(featured_df)

featured_df = featured_df.dropna()

print(f"Feature engineering complete: {len(featured_df)} records, {len(featured_df.columns)} features")

In [ ]:
# Normalize features using saved scaling parameters
feature_cols = [col for col in featured_df.columns if col not in ['Date']]
normalized_df, _ = preprocessor.normalize_features(
    featured_df[feature_cols], method=Config.NORMALIZATION_METHOD
)

print("Features normalized")

## 3. Single-Step Predictions

In [ ]:
# Prepare input for single-step prediction
if metadata.model_type in ['LSTM', 'GRU']:
    # Use last sequence_length records
    input_features = normalized_df.values[-Config.SEQUENCE_LENGTH:]
    input_features = input_features.reshape(1, Config.SEQUENCE_LENGTH, -1)
else:
    # Use last record
    input_features = normalized_df.values[-1:]

# Generate single-step prediction
prediction = predictor.predict_single_step(model, input_features)

print(f"Single-step prediction (normalized): {prediction}")

# Denormalize (simplified - in practice use saved scaling params)
last_close = featured_df['Close'].iloc[-1]
print(f"\nCurrent gold price: ${last_close:.2f}")
print(f"Predicted next price: ${prediction * last_close:.2f} (approximate)")

## 4. Multi-Step Forecasts

In [ ]:
# Generate multi-step predictions
horizons = [7, 14, 30]  # Different forecast horizons

forecasts = {}
for horizon in horizons:
    print(f"Generating {horizon}-day forecast...")
    predictions = predictor.predict_multi_step(model, input_features, horizon, metadata.model_type)
    forecasts[horizon] = predictions
    print(f"  Generated {len(predictions)} predictions")

print("\nForecasts complete!")

In [ ]:
# Create forecast dataframes with dates
last_date = featured_df['Date'].iloc[-1]
forecast_dfs = {}

for horizon, predictions in forecasts.items():
    dates = [last_date + timedelta(days=i+1) for i in range(len(predictions))]
    forecast_dfs[horizon] = pd.DataFrame({
        'Date': dates,
        'Predicted_Close': predictions
    })
    
print("Sample 7-day forecast:")
print(forecast_dfs[7].head())

## 5. Prediction Confidence Intervals

In [ ]:
# Compute confidence intervals for 30-day forecast
horizon = 30
predictions_30d = forecasts[horizon]

# Calculate confidence intervals
confidence_intervals = predictor.compute_prediction_intervals(
    predictions_30d, confidence=Config.CONFIDENCE_LEVEL
)

print(f"Confidence intervals at {Config.CONFIDENCE_LEVEL*100}% level:")
print(f"Shape: {confidence_intervals.shape}")
print(f"Range: [{confidence_intervals[:, 0].min():.4f}, {confidence_intervals[:, 1].max():.4f}]")

In [ ]:
# Create dataframe with confidence intervals
dates_30d = [last_date + timedelta(days=i+1) for i in range(horizon)]

forecast_with_ci = pd.DataFrame({
    'Date': dates_30d,
    'Predicted_Close': predictions_30d,
    'Lower_CI': confidence_intervals[:, 0],
    'Upper_CI': confidence_intervals[:, 1]
})

print("\n30-day forecast with confidence intervals:")
print(forecast_with_ci.head(10))

## 6. Visualize Predictions

In [ ]:
# Plot historical data with 30-day forecast
lookback = 90  # Show last 90 days of history
historical = featured_df[['Date', 'Close']].iloc[-lookback:]

fig, ax = plt.subplots(figsize=(15, 7))

# Historical prices
ax.plot(historical['Date'], historical['Close'], 
        label='Historical', linewidth=2, color='steelblue')

# Forecasted prices (denormalized approximately)
# Note: In practice, use proper denormalization with saved scaling params
scale_factor = historical['Close'].mean()
ax.plot(forecast_with_ci['Date'], forecast_with_ci['Predicted_Close'] * scale_factor,
        label='Forecast', linewidth=2, color='orange', linestyle='--')

# Confidence intervals
ax.fill_between(forecast_with_ci['Date'], 
                forecast_with_ci['Lower_CI'] * scale_factor,
                forecast_with_ci['Upper_CI'] * scale_factor,
                alpha=0.3, color='orange', label=f'{Config.CONFIDENCE_LEVEL*100}% CI')

ax.set_title('Gold Price Forecast with Confidence Intervals', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare different forecast horizons
fig, ax = plt.subplots(figsize=(15, 7))

# Historical
ax.plot(historical['Date'], historical['Close'],
        label='Historical', linewidth=2, color='steelblue')

# Different horizons
colors = ['orange', 'green', 'red']
for i, (horizon, forecast_df) in enumerate(forecast_dfs.items()):
    ax.plot(forecast_df['Date'], forecast_df['Predicted_Close'] * scale_factor,
            label=f'{horizon}-day Forecast', linewidth=2, 
            color=colors[i], linestyle='--', marker='o', markersize=3)

ax.set_title('Gold Price Forecasts - Multiple Horizons', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Feature Importance (for tree-based models)

In [ ]:
if metadata.model_type in ['XGBoost', 'RandomForest']:
    # Get feature importance
    importance = model.feature_importances_
    feature_names = metadata.feature_list[1:]  # Exclude Date
    
    # Create dataframe
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importance
    }).sort_values('Importance', ascending=False)
    
    # Plot top 20 features
    fig, ax = plt.subplots(figsize=(10, 8))
    top_features = importance_df.head(20)
    ax.barh(top_features['Feature'], top_features['Importance'])
    ax.set_xlabel('Importance')
    ax.set_title('Top 20 Feature Importances', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print("Top 10 most important features:")
    print(importance_df.head(10))
else:
    print(f"Feature importance not available for {metadata.model_type} models")

## 8. Generate Comprehensive Prediction Report

In [ ]:
# Use VisualizationManager to create report
viz_manager = VisualizationManager()

# Prepare data for report
actual_data = historical
predictions_data = forecast_with_ci.copy()
predictions_data['Close'] = predictions_data['Predicted_Close'] * scale_factor

# Create comprehensive report
report_path = viz_manager.create_prediction_report(
    predictions=predictions_data,
    actual=actual_data,
    confidence_intervals=forecast_with_ci[['Lower_CI', 'Upper_CI']] * scale_factor,
    metadata=metadata,
    output_format='html',
    report_name='notebook_prediction_report'
)

print(f"Comprehensive report saved to: {report_path}")

## 9. Prediction Statistics

In [ ]:
# Calculate prediction statistics
pred_stats = {
    'Mean Prediction': predictions_30d.mean(),
    'Min Prediction': predictions_30d.min(),
    'Max Prediction': predictions_30d.max(),
    'Std Prediction': predictions_30d.std(),
    'Trend': 'Upward' if predictions_30d[-1] > predictions_30d[0] else 'Downward',
    'Total Change (%)': ((predictions_30d[-1] - predictions_30d[0]) / predictions_30d[0] * 100)
}

print("30-Day Forecast Statistics:")
print("="*60)
for stat, value in pred_stats.items():
    if isinstance(value, str):
        print(f"{stat:<25} {value}")
    else:
        print(f"{stat:<25} {value:.4f}")
print("="*60)

## Summary

This notebook demonstrated:
1. ✅ Loading trained models from registry
2. ✅ Generating single-step predictions
3. ✅ Generating multi-step forecasts (7, 14, 30 days)
4. ✅ Computing confidence intervals at 95% level
5. ✅ Visualizing predictions with uncertainty bands
6. ✅ Comparing different forecast horizons
7. ✅ Feature importance analysis (for tree models)
8. ✅ Generating comprehensive prediction reports

**Key Features:**
- Prediction Service loads models and generates forecasts
- Supports both sequence models (LSTM/GRU) and tree models (XGBoost/RF)
- Confidence intervals provide uncertainty quantification
- Multiple visualization options for different use cases

**Usage Tips:**
- Always use the same preprocessing and feature engineering as training
- Confidence intervals widen for longer forecast horizons
- Monitor model performance over time and retrain when drift detected
- Compare predictions across different models for robustness